In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigs

In [3]:
# Extraemos los datos de la web de la Universidad de Stanford.
url = "https://snap.stanford.edu/data/soc-sign-bitcoinotc.csv.gz"
columns = ['start', 'end','weight','time']
df = pd.read_csv(url,names = columns)

In [4]:
# Hacemos una biyección que mapeé los IDs a índices contiguos. 
unique_nodes = pd.unique(df[['start','end']].values.ravel('K'))
node_map = {node_id: i for i, node_id in enumerate(unique_nodes)}
inverse_node_map = {i: node_id for i, node_id in enumerate(unique_nodes)}

df['i_start'] = df['start'].map(node_map)
df['i_end'] = df['end'].map(node_map)

N = len(unique_nodes)

In [5]:
# Calculamos la matriz ponderada con los pesos de las aristas dirigidas. 
I = df['i_start'].values
J = df['i_end'].values
w = df['weight'].values

A = coo_matrix((w, (I,J)), shape=(N,N))
print(f"Dimensión (nodos): {N} x {N}")
print(f"Número de aristas dirigidas: {A.nnz}")

Dimensión (nodos): 5881 x 5881
Número de aristas dirigidas: 35592


In [6]:
# Calculamos los autovalores y autovectores a izquierda y derecha de A y nos quedamos con las partes reales de estos. 
lambda_R, x = eigs(A,k=1,which='LM')
lambda_L, y = eigs(A.transpose(),k=1,which='LM')

u = np.real(x[:,0])
v = np.real(y[:,0])

# Asociamos cada coordenada al ID de su vértice y hacemos el DataFrame correspondiente al espectro de A. 
results =[]
for k in range(N):
    results.append((inverse_node_map[k],u[k],v[k]))

df_spectrum = pd.DataFrame(results, columns =['ID','Issuing Authority','Receiving Authority'])

df_negsink = df_spectrum.sort_values(by='Receiving Authority',ascending=True)
df_possink = df_spectrum.sort_values(by='Receiving Authority',ascending=False)

print("--- Extremo Inferior del Espectro Receptor ---")
print(df_negsink.head(10))
print("\n--- Extremo superior del Espectro Receptor ---")
print(df_possink.head(10))

--- Extremo Inferior del Espectro Receptor ---
        ID  Issuing Authority  Receiving Authority
3218  3744          -0.152716            -0.170288
1780  2017           0.004190            -0.139880
4009  4733          -0.032458            -0.114390
3967  4666          -0.059373            -0.109464
3966  4661          -0.063749            -0.109400
3850  4531          -0.071285            -0.102781
3953  4654          -0.075645            -0.101418
3231  3760          -0.202826            -0.101296
4008  4707          -0.019123            -0.099135
3982  4683          -0.061855            -0.091126

--- Extremo superior del Espectro Receptor ---
        ID  Issuing Authority  Receiving Authority
1618  1810           0.192886             0.320444
3347  3897           0.067031             0.266026
3937  4635           0.013100             0.186356
870    905           0.146129             0.153198
3566  4172           0.080538             0.149453
1803  2045           0.142609         

Tras haber radiografiado la topología de la red, podemos observar que el ID 3744 es un posible estafador activo pues es el mínimo global receptor (-0.170) pero además también cuenta con una valoración fuertemente negativa como emisor (-0.152). 
A continuación investigaremos las interacciones de este usuario con el resto de la red para comprobar la veracidad de nuestra hipótesis. 

In [7]:
target_id = 3744

# Extraemos los vértices con los que se relaciona el 3744.  
E_in = df[df['end'] == target_id]
wsum_in = E_in['weight'].sum()
numars_in = len(E_in)

E_out = df[df['start'] == target_id]
wsum_out = E_out['weight'].sum()
numars_out = len(E_out)

print(f"--- Análisis Topológico Local del Vértice {target_id}")
print(f"Cardinalidad de N^-(v): {numars_in} aristas incidentes")
print(f"Grado ponderado de entrada (deg_in): {wsum_in}")

print(f"Cardinalidad de N^+(v): {numars_out} aristas emitidas")
print(f"Grado ponderado de salida (deg_out): {wsum_out}")

print(f"\n --- Extracto del conjunto E_in para el vértice {target_id} ---")
print(E_in[['start','weight']].head(10))

--- Análisis Topológico Local del Vértice 3744
Cardinalidad de N^-(v): 81 aristas incidentes
Grado ponderado de entrada (deg_in): -675
Cardinalidad de N^+(v): 32 aristas emitidas
Grado ponderado de salida (deg_out): 80

 --- Extracto del conjunto E_in para el vértice 3744 ---
       start  weight
19932   2962      10
19945   1802     -10
19946   1363     -10
19947   2658     -10
19949   2296     -10
19951   2647     -10
19957   2045     -10
19959   2028     -10
19999   2252      -5
20013   1948     -10


Dado que el vértice con ID 2962 destaca entre el resto por dar una confianza tan positiva al 3744, estudiamos la interacción entre estos dos usuarios y otros posibles sospechosos. 

In [8]:
# Calculamos la matriz de adyacencia para los casos en los que se de máxima confianza. 
df_trust = df[df['weight'] == 10].copy()
M_files, M_columns = df_trust['i_start'], df_trust['i_end']
M_values = np.ones(len(df_trust))

M = coo_matrix((M_values,(M_files,M_columns)), shape=(N,N))
M = M.tocsr()

i_target = node_map[3744]
i_suspect = node_map[2962]

# Calculamos la matriz de los caminos de orden 2
M2 = M.dot(M)

go_ar = M[i_target,i_suspect]
back_ar = M[i_suspect,i_target]

cycles_target = M2[i_target, i_target]
cycles_suspect = M2[i_suspect, i_suspect]

print("--- Evaluación de la matriz de adyacencia directa M ---")
print(f"Existencia de arista 2962 -> 3744 (m_ij): {int(go_ar)}")
print(f"Existencia de arista 3744 -> 2962 (m_ij): {int(back_ar)}")

print("--- Evaluación de la matriz de ciclos y caminos M^2 ---")
print(f"Caminos cerrados de longitud 2 para el nodo 3744: {int(cycles_target)}")
print(f"Caminos cerrados de longitud 2 para el nodo 2962: {int(cycles_suspect)}")

--- Evaluación de la matriz de adyacencia directa M ---
Existencia de arista 2962 -> 3744 (m_ij): 1
Existencia de arista 3744 -> 2962 (m_ij): 1
--- Evaluación de la matriz de ciclos y caminos M^2 ---
Caminos cerrados de longitud 2 para el nodo 3744: 4
Caminos cerrados de longitud 2 para el nodo 2962: 1


In [ ]:
# Calculamos el producto de Hadamard de M por su traspuesta
C = M.multiply(M.transpose())

file_target = C[i_target, :]

i_party = file_target.nonzero()[1]
suspects = []
for i in i_party: 
    x = inverse_node_map[i]
    suspects.append(x)

weights_intersection = file_target.data

print(f"---Análisis del Anillo de Colusión para el Estafador {target_id}---")

if len(i_party) == 0:
    print("El subespacio es vacío. No hay colusión bidireccional.")
else:
    for i,idx in enumerate(suspects):
        weight = weights_intersection[i]
        print(f"Cómplice bidireccional encontrado: ID original {suspects[i]}")
        print(f"-> Magnitud de la singularidad: {int(weight)}")

---Análisis del Anillo de Colusión para el Estafador 3744---
Cómplice bidireccional encontrado: ID original 2962
-> Magnitud de la singularidad: 1
Cómplice bidireccional encontrado: ID original 3756
-> Magnitud de la singularidad: 1
Cómplice bidireccional encontrado: ID original 3759
-> Magnitud de la singularidad: 1
Cómplice bidireccional encontrado: ID original 3760
-> Magnitud de la singularidad: 1


In [10]:
print("---Auditoría Topológica Estricta---")

for x in suspects:
    i = node_map[x]

    m_go = M[i, i_target]
    m_back = M[i_target, i]

    c = m_go * m_back
    cm = C[i_target,i]

    print(f"ID {x}")
    print(f"  M[{x} -> 3744] (Ida) = {int(m_go)}")
    print(f"  M[3744 -> {x}] (Vuelta) = {int(m_back)}")
    print(f"  Hadamard Teórico (ida * vuelta) = {int(c)}")
    print(f"  Valor real almacenado en C = {int(cm)}\n")

---Auditoría Topológica Estricta---
ID 2962
  M[2962 -> 3744] (Ida) = 1
  M[3744 -> 2962] (Vuelta) = 1
  Hadamard Teórico (ida * vuelta) = 1
  Valor real almacenado en C = 1

ID 3756
  M[3756 -> 3744] (Ida) = 1
  M[3744 -> 3756] (Vuelta) = 1
  Hadamard Teórico (ida * vuelta) = 1
  Valor real almacenado en C = 1

ID 3759
  M[3759 -> 3744] (Ida) = 1
  M[3744 -> 3759] (Vuelta) = 1
  Hadamard Teórico (ida * vuelta) = 1
  Valor real almacenado en C = 1

ID 3760
  M[3760 -> 3744] (Ida) = 1
  M[3744 -> 3760] (Vuelta) = 1
  Hadamard Teórico (ida * vuelta) = 1
  Valor real almacenado en C = 1

